In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# cd '/Users/peiyingw/526 LLM/FinalProject'

## Split training data into training and validation dataset

In [ ]:
label = pd.read_csv('test_labels.csv')
fil = label[label['toxic'] != -1] 
fil.shape

In [ ]:
tmp = pd.read_csv('test.csv')

In [ ]:
test = pd.read_csv('test.csv')
test_fil = test[test['id'].isin(fil['id'])]
test_fil.shape

In [ ]:
train = pd.read_csv('train.csv')
train.shape

In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train,
    test_size=0.2,
    stratify=train['toxic'],
    random_state=42
)

train_data

In [ ]:
val_data

In [ ]:
train_data.to_csv("train_data.csv")

In [ ]:
val_data.to_csv("val_data.csv")

## Computing the estimation of training the model.

In [ ]:
!pip install transformers datasets accelerate peft bitsandbytes trl

In [ ]:
from huggingface_hub import login
# login = 'xxxx'

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
texts = train_data["comment_text"].tolist()

tokenized = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt"
)

avg_len = tokenized["input_ids"].ne(tokenizer.pad_token_id).sum(dim=1).float().mean()
print("Average token length:", avg_len.item())
print("Total tokens:", tokenized["input_ids"].ne(tokenizer.pad_token_id).sum().item())

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import time

def tokenize_fn(example):
    out = tokenizer(
        example["comment_text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized_train = small_train.map(tokenize_fn, batched=False)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

args = TrainingArguments(
    output_dir="./calibration_run",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    data_collator=data_collator
)

torch.cuda.reset_peak_memory_stats()
start = time.time()

trainer.train()

end = time.time()

elapsed = end - start
peak_mem = torch.cuda.max_memory_allocated() / 1024**3

print("Elapsed seconds:", elapsed)
print("Peak GPU memory GB:", peak_mem)

In [ ]:
estimated_hours = elapsed * (127000 / len(small_train)) * 3 / 3600
print("Estimated full QLoRA H200-hours:", estimated_hours)